# Trace a multi-tool LangGraph agent

Follow a tool-calling agent from live logs to a visual trace without an API key. The agent will look up a notebook's price, calculate the subtotal for four notebooks, and apply a 10% discount. The expected total is **$45.00**.

A scripted chat model makes the example reproducible; LangGraph, its tool execution, callbacks and Ollie's recording are real. The model is not reasoning about arbitrary shopping questions. The last cell creates a VizTracer report to open in your browser, without collecting a second Python profile.

First install the optional dependencies from the repository root:
```bash
uv sync --group dev --group notebooks
uv run --no-sync jupyter lab notebooks/01_langgraph_multi_tool_trace.ipynb
```
Then run the cells in order using the project environment's Python kernel. See the [notebook guide](README.md) for viewer controls and privacy guidance.

## 1. Send application logs to cell output

A dedicated logger writes to the notebook's current stdout stream. Each model and tool message is emitted immediately; the execution cell will not wait until the trace is exported to show progress. These are application logs, separate from Ollie's structured trace records.

In [1]:
import json
import logging
import os
import sys
from pathlib import Path
from uuid import uuid4

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_ollie import CapturePolicy, RecordingSession, SpanKind, SpanStatus

# Keep this example local even if the shell has hosted tracing enabled.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

logger = logging.getLogger("ollie.notebooks.multi_tool")
logger.setLevel(logging.INFO)
logger.propagate = False
# Re-running this cell replaces only this example's handlers.
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()
log_handler = logging.StreamHandler(sys.stdout)
log_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(log_handler)
print("Ready: application logs will appear in the execution cell.")

Ready: application logs will appear in the execution cell.


## 2. Define three observable tools

Each tool does one small job and logs its input and result. All data is synthetic and stays in this process. Separating the work makes it possible to see whether latency or repeated calls come from price lookup, arithmetic or discount handling.

In [2]:
@tool
def get_unit_price(sku: str) -> float:
    """Look up a synthetic catalog item's unit price in USD."""
    logger.info("TOOL get_unit_price start: sku=%s", sku)
    price = {"notebook": 12.50}[sku]
    logger.info("TOOL get_unit_price result: $%.2f", price)
    return price


@tool
def calculate_subtotal(unit_price: float, quantity: int) -> float:
    """Multiply a USD unit price by a positive item quantity."""
    if quantity < 1:
        raise ValueError("quantity must be positive")
    logger.info("TOOL calculate_subtotal start: $%.2f x %d", unit_price, quantity)
    subtotal = round(unit_price * quantity, 2)
    logger.info("TOOL calculate_subtotal result: $%.2f", subtotal)
    return subtotal


@tool
def apply_discount(subtotal: float, percent: float) -> float:
    """Apply a percentage discount to a USD subtotal."""
    if not 0 <= percent <= 100:
        raise ValueError("discount must be between 0 and 100")
    logger.info("TOOL apply_discount start: $%.2f, %.0f%% off", subtotal, percent)
    total = round(subtotal * (1 - percent / 100), 2)
    logger.info("TOOL apply_discount result: $%.2f", total)
    return total


tools = [get_unit_price, calculate_subtotal, apply_discount]

## 3. Use a scripted model that consumes real tool results

The model requests one tool at a time. Its next request uses the previous tool's actual return value, and its final answer uses the discounted total. This exercises a real model → tool → model loop while avoiding provider availability, credentials and nondeterministic answers.

The model reports illustrative token counts so the trace has usage counters. They are not measured provider usage or a cost estimate. To trace your own agent later, replace this model with a supported tool-calling chat model and keep the recording callback integration unchanged.

In [3]:
class LocalQuoteModel(BaseChatModel):
    @property
    def _llm_type(self) -> str:
        return "scripted-local-quote"

    def bind_tools(self, tools, **kwargs):
        names = {item.name for item in tools}
        if names != {"get_unit_price", "calculate_subtotal", "apply_discount"}:
            raise ValueError("the example requires its three catalog tools")
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        previous = next(
            (message for message in reversed(messages) if isinstance(message, ToolMessage)),
            None,
        )
        if previous is None:
            name, args = "get_unit_price", {"sku": "notebook"}
        elif previous.name == "get_unit_price":
            name, args = (
                "calculate_subtotal",
                {"unit_price": float(previous.content), "quantity": 4},
            )
        elif previous.name == "calculate_subtotal":
            name, args = "apply_discount", {"subtotal": float(previous.content), "percent": 10}
        elif previous.name == "apply_discount":
            total = float(previous.content)
            logger.info("MODEL final answer: $%.2f", total)
            answer = AIMessage(content=f"Total: ${total:.2f} (4 notebooks, 10% discount).")
            answer.usage_metadata = {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15}
            return ChatResult(generations=[ChatGeneration(message=answer)])
        else:
            raise ValueError("unexpected tool response")

        logger.info("MODEL requests %s", name)
        response = AIMessage(
            content="",
            tool_calls=[{"name": name, "args": args, "id": f"call-{name}"}],
            usage_metadata={"input_tokens": 10, "output_tokens": 5, "total_tokens": 15},
        )
        return ChatResult(generations=[ChatGeneration(message=response)])

## 4. Build the LangGraph tool-calling loop

The model node produces either a tool request or a final answer. `tools_condition` routes requests to `ToolNode`; tool results return to the model. Passing the runtime config into the model preserves callback parentage, so the trace contains the graph, its nodes, chat-model calls and tool calls.

In [4]:
def build_agent():
    model = LocalQuoteModel().bind_tools(tools)

    def call_model(state: MessagesState, config: RunnableConfig):
        return {"messages": [model.invoke(state["messages"], config=config)]}

    graph = StateGraph(MessagesState)
    graph.add_node("model", call_model)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition, {"tools": "tools", END: END})
    graph.add_edge("tools", "model")
    return graph.compile(name="NotebookQuoteAgent")

## 5. Run the agent and record its execution

Watch this cell's output: model requests, tool start/result logs and streamed graph updates should alternate before the final answer. No sleeps or background log replay are involved.

Bounded content capture is enabled only because these inputs and outputs are synthetic. For private workloads, start with `CapturePolicy.metadata()` and review the [capture guidance](../README.md#safety-and-recording-limits) before retaining bodies. The stream must finish before the session closes. Re-running this cell creates a fresh agent, session and artifact directory.

In [5]:
artifact_dir = Path("notebook-artifacts") / f"multi-tool-{uuid4().hex[:12]}"
artifact_dir.mkdir(parents=True, exist_ok=False)
trace_path = artifact_dir / "agent_trace.json"
final_answer = None
logger.info("RUN starting local quote agent")
with RecordingSession(
    capture=CapturePolicy.bounded(max_depth=12, max_string_length=2000),
    service_name="notebook-quote-agent",
) as recording:
    agent = build_agent()
    for update in agent.stream(
        {"messages": [("user", "What is the total for 4 notebooks with 10% off?")]},
        config={"callbacks": [recording.handler], "recursion_limit": 16},
        stream_mode="updates",
    ):
        for node, changed_state in update.items():
            logger.info("STREAM completed node: %s", node)
            message = changed_state["messages"][-1]
            if isinstance(message, AIMessage) and not message.tool_calls:
                final_answer = message.content

recording.export(trace_path, pretty=True)
assert final_answer == "Total: $45.00 (4 notebooks, 10% discount)."
logger.info("RUN finished: %s", final_answer)
print(f"Trace saved to: {trace_path.resolve()}")

[INFO] RUN starting local quote agent
[INFO] MODEL requests get_unit_price
[INFO] STREAM completed node: model
[INFO] TOOL get_unit_price start: sku=notebook
[INFO] TOOL get_unit_price result: $12.50
[INFO] STREAM completed node: tools
[INFO] MODEL requests calculate_subtotal
[INFO] STREAM completed node: model
[INFO] TOOL calculate_subtotal start: $12.50 x 4
[INFO] TOOL calculate_subtotal result: $50.00
[INFO] STREAM completed node: tools
[INFO] MODEL requests apply_discount
[INFO] STREAM completed node: model
[INFO] TOOL apply_discount start: $50.00, 10% off
[INFO] TOOL apply_discount result: $45.00
[INFO] STREAM completed node: tools
[INFO] MODEL final answer: $45.00
[INFO] STREAM completed node: model
[INFO] RUN finished: Total: $45.00 (4 notebooks, 10% discount).
Trace saved to: /home/charilaos/Workspace/langgraph_instrumentation/notebooks/notebook-artifacts/multi-tool-619d7a9037a1/agent_trace.json


## 6. Check the evidence before interpreting the picture

A correct answer alone does not prove that all three tools ran. Inspect the immutable recording snapshot and confirm one successful span per tool, four model calls and no reported recording loss. Tool durations are elapsed wall time, not CPU attribution.

In [6]:
bundles = recording.snapshot()
spans = [span for bundle in bundles for span in bundle.spans]
tool_spans = [span for span in spans if span.kind == SpanKind.TOOL]
model_spans = [span for span in spans if span.kind == SpanKind.LLM]
assert len(bundles) == 1
assert len(tool_spans) == 3 and len(model_spans) == 4
assert all(span.status == SpanStatus.SUCCESS for span in spans)
assert not any(recording.diagnostics.values()), recording.diagnostics
for span in tool_spans:
    print(f"{span.name}: {span.duration_ns / 1_000_000:.3f} ms ({span.status.value})")
print(f"Recorded {len(spans)} spans; {len(model_spans)} model calls; no recording loss.")
trace_payload = json.loads(trace_path.read_text(encoding="utf-8"))
assert trace_payload["traceEvents"]
assert not trace_payload["otherData"]["incomplete"]

Tool: get_unit_price: 1.065 ms (success)
Tool: calculate_subtotal: 0.840 ms (success)
Tool: apply_discount: 0.753 ms (success)
Recorded 19 spans; 4 model calls; no recording loss.


## 7. Open the VizTracer report in your browser

Ollie's export uses Trace Event JSON, which VizTracer's report builder can display. We turn that existing trace into an HTML report and provide a browser link instead of embedding the timeline in notebook output. This avoids VS Code notebook layout and theme issues. We do not enable VizTracer's function profiler or merge separate timelines. `file_info=False` avoids collecting Python source files into the report.

Click the report link below. If VS Code blocks a local-file link or opens it in an editor, copy the printed `file://` URI into your browser. You can also open the printed HTML path directly from your file manager. The notebook does not launch a browser or background server automatically, so **Run All** also works in headless environments.

If you prefer an HTTP link, run the printed loopback-server command in a terminal, then open `http://127.0.0.1:9001/agent_trace.html`. Stop the server with **Ctrl+C**. For SSH, containers or Codespaces, privately forward port 9001 through VS Code; a file URI on the remote kernel is not a local file on your computer. Do not expose the report server publicly.

Only open trusted reports: the viewer is active JavaScript and loads compatibility/highlighting scripts from public CDNs. Moving it to a browser does not remove those dependencies. Use this synthetic example, not private traces, unless your environment approves those scripts. For fully offline viewing, open the same JSON in an [offline Perfetto viewer](https://perfetto.dev/docs/visualization/offline).

In [7]:
import shlex

from IPython.display import Markdown, display
from viztracer.report_builder import ReportBuilder

html_path = artifact_dir / "agent_trace.html"
agent_trace_json = artifact_dir / 'agent_trace.json'
ReportBuilder(trace_payload, verbose=0).save(str(html_path), file_info=False)
report_uri = html_path.resolve().as_uri()

# display(Markdown(f"[Open the VizTracer report in your browser]({report_uri})"))
vizviewer_cmd = f'vizviewer {agent_trace_json.absolute()}'
print(f"run the following command to start the interactive trace viewer:\n\n {vizviewer_cmd}")

run the following command to start the interactive trace viewer:

 vizviewer /home/charilaos/Workspace/langgraph_instrumentation/notebooks/notebook-artifacts/multi-tool-619d7a9037a1/agent_trace.json


## For non-restricted environments
If you are not firewall-restricted to access the public perfetto website, you can simply run the official bash one-liner of Perfetto
to view the trace:

In [ ]:
!curl -s -O https://raw.githubusercontent.com/google/perfetto/main/tools/open_trace_in_ui && chmod +x open_trace_in_ui && ./open_trace_in_ui -i {agent_trace_json.absolute()}

----

## One liner execution for local vizviewer 
Note that this might not be installable in restricted enterprise environments

The following starts a local server that serves the ollie-perfetto trace produce above.

In [ ]:
# run the following cell (will depend on port 9001 being open to start vizviewer)
!{vizviewer_cmd}

## What to investigate next

Click inside the viewer, use **W/S** to zoom and **A/D** to pan, and select a tool slice to inspect its captured attributes. Find the three tool operations and the four model calls around them. Containment arrows show nesting, not a proven critical path.

The execution logs explain the example's progress; the trace supplies structured timing, parentage and usage evidence. To optimize a real agent, look for redundant tool round trips, oversized context or unnecessary model calls, then compare changes while checking answer quality. See the [evaluation guide](../docs/evals.md) for checks over tool calls and execution budgets, or the [architecture overview](../docs/architecture.md) for the recording pipeline.